# Bodrum Hotel & Destination Intelligence
## 01 - Data Collection

Bu proje, Bodrum'daki konaklama tesislerinin genel özelliklerini, müşteri memnuniyetini,
fiyat göstergelerini ve destinasyon farklılıklarını ilerleyen aşamalarda incelemek üzere
tasarlanmıştır.

Bu notebook yalnızca mevcut ham veri dosyalarını projeye dahil eder, görevlerini açıklar ve
temel yapılarını inceler. Bu aşamada **veri temizliği, düzeltme, özellik mühendisliği veya
modelleme yapılmaz**.

### 1. Kütüphaneler

Temel tablo işlemleri için `pandas` ve `numpy`, güvenli ve platformdan bağımsız dosya yolları
için `pathlib` kullanılır. Görüntüleme ayarları yalnızca notebook okunabilirliğini artırır.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

### 2. Proje dosya yolları

Notebook farklı çalışma dizinlerinden açılabileceği için dosyalar proje kökü, `data/` ve
`data/raw/` olasılıklarında aranır. Zorunlu bir dosya bulunamazsa aranan konumları içeren
anlaşılır bir hata üretilir.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

def find_project_file(file_name, required=True):
    roots = [PROJECT_ROOT, PROJECT_ROOT.parent]
    candidates = []
    for root in roots:
        candidates.extend([root / file_name, root / "data" / file_name, root / "data" / "raw" / file_name])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    if required:
        searched = "\n".join(f"- {path}" for path in candidates)
        raise FileNotFoundError(f"'{file_name}' bulunamadı. Aranan konumlar:\n{searched}")
    return None

FILE_ROLES = {
    "bodrum_hotels_master_2026-08-24.csv": "Main hotel-level dataset",
    "bodrum_hotels_master_2026-08-24.xlsx": "Optional Excel copy of the master dataset",
    "bodrum_hotels_master_README.txt": "Dataset documentation",
}

file_paths = {
    name: find_project_file(name, required=name != "bodrum_hotels_master_2026-08-24.xlsx")
    for name in FILE_ROLES
}

### 3. Dosya envanteri

Aşağıdaki envanter her beklenen dosyanın türünü, bulunma durumunu, otomatik hesaplanan
boyutunu ve projedeki rolünü gösterir. Excel kopyası isteğe bağlıdır; ana analiz kaynağı CSV'dir.

In [3]:
inventory = pd.DataFrame([
    {
        "file_name": name,
        "file_type": Path(name).suffix.lower().lstrip("."),
        "exists": path is not None and path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path is not None and path.exists() else np.nan,
        "role": FILE_ROLES[name],
    }
    for name, path in file_paths.items()
])
inventory

,file_name,file_type,exists,size_kb,role
0,bodrum_hotels_master_2026-08-24.csv,csv,True,76.38,Main hotel-level dataset
1,bodrum_hotels_master_2026-08-24.xlsx,xlsx,True,35.60,Optional Excel copy of the master dataset
2,bodrum_hotels_master_README.txt,txt,True,0.69,Dataset documentation


### 4. Ana veri setinin yüklenmesi

Otel seviyesindeki ana CSV yüklenir. Telefon numarası nicel bir ölçüm değil, kimlik niteliğinde
metin olduğundan baştaki `+` işaretini korumak için metin olarak okunur. Bu bölüm yalnızca boyut,
örnek satırlar ve kolon adlarını gösterir; veri üzerinde değişiklik yapılmaz.

In [4]:
MASTER_PATH = file_paths["bodrum_hotels_master_2026-08-24.csv"]
df = pd.read_csv(MASTER_PATH, dtype={"phone": "string"})
df["area_hotel_count"] = df.groupby("area")["hotel_id"].transform("size")

dataset_dimensions = pd.DataFrame({"metric": ["rows", "columns"], "value": df.shape})
display(dataset_dimensions)
display(pd.DataFrame({"column": df.columns}))
display(df.head())
display(df.tail())

,metric,value
0,rows,192
1,columns,20


,column
0,hotel_id
1,place_id
2,hotel_name
3,area
4,district
5,province
6,country
7,property_category
8,official_star_rating
9,google_rating


,hotel_id,place_id,hotel_name,area,district,province,country,property_category,official_star_rating,google_rating,google_rating_scale,google_review_count,search_price_usd_snapshot,price_note,address,phone,business_status,collected_at,source_url,area_hotel_count
0,BOD001,ChIJAxKJBV4MvhQRxCb482IcG0M,Aksoy Taş Ev,Akyarlar,Bodrum,Muğla,Türkiye,Otel,NaN,4.70,5,135,194.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Akyarlar, Tarik Akan Sk NO:15, 48960 Bodrum/Muğla, Türkiye",+905414124141,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Aksoy%20Ta%C5%9F%20Ev&query_place_id=ChIJAxKJBV4...,16
1,BOD002,ChIJqWC7b3YMvhQRCR9qlsxFzmQ,Armonia Holiday Village & Spa,Akyarlar,Bodrum,Muğla,Türkiye,Otel,NaN,4.20,5,3197,191.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Turgutreis, Kemer Mevkii, 48400 Bodrum/Muğla, Türkiye",+902529991303,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Armonia%20Holiday%20Village%20%26%20Spa&query_pl...,16
2,BOD003,ChIJS8TbUlYNvhQR_fh-bq0pJDg,ASPAT HOTEL BODRUM Beach Restaurant,Akyarlar,Bodrum,Muğla,Türkiye,Otel,NaN,4.30,5,246,57.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Akyarlar, atatürk akyarlar caddesi no 253, 48960 Bodrum/Muğla, Türkiye",+905308656175,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=ASPAT%20HOTEL%20BODRUM%20Beach%20Restaurant&quer...,16
3,BOD004,ChIJXUoh1okLvhQR-k-X9ouGc3c,Bendis Beach Hotel,Akyarlar,Bodrum,Muğla,Türkiye,Otel,NaN,3.90,5,2469,104.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Akyarlar, Kemer Mevkii No: 24, 48960 Bodrum/Muğla, Türkiye",+902523210924,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Bendis%20Beach%20Hotel&query_place_id=ChIJXUoh1o...,16
4,BOD005,ChIJdTwebV4MvhQRVC_Shj1Cjqo,ByAkkan,Akyarlar,Bodrum,Muğla,Türkiye,Otel,NaN,4.40,5,203,84.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Akyarlar, Tarik Akan Sk No:33, 48000 Bodrum/Muğla, Türkiye",+905324996884,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=ByAkkan&query_place_id=ChIJdTwebV4MvhQRVC_Shj1Cjqo,16


,hotel_id,place_id,hotel_name,area,district,province,country,property_category,official_star_rating,google_rating,google_rating_scale,google_review_count,search_price_usd_snapshot,price_note,address,phone,business_status,collected_at,source_url,area_hotel_count
187,BOD188,ChIJ_2z-SKNtvhQR0aTUt1j8QJk,Tangiers Hotel Yalıkavak,Yalıkavak,Bodrum,Muğla,Türkiye,Otel,NaN,4.10,5,85,196.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Yalıkavak, Çarşı Cd. No:42, 48990 Bodrum/Muğla, Türkiye",+902522771266,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Tangiers%20Hotel%20Yal%C4%B1kavak&query_place_id...,22
188,BOD189,ChIJzZ-AXwhxvhQRIcUswS4ooeI,The Bodrum EDITION,Yalıkavak,Bodrum,Muğla,Türkiye,Otel,NaN,4.40,5,1016,841.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Dirmil, Balyek Cd. No 5A, 48400 Bodrum/Muğla, Türkiye",+902523113131,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=The%20Bodrum%20EDITION&query_place_id=ChIJzZ-AXw...,22
189,BOD190,ChIJ3Z4rPURxvhQRecoeZJ3euTg,The Sign Highlight Hotel,Yalıkavak,Bodrum,Muğla,Türkiye,Otel,NaN,4.30,5,125,289.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Merkez Mah, Tilkicik Cd. No:182 Sk No:3, 48990 Bodrum/Muğla, Türkiye",+902526111010,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=The%20Sign%20Highlight%20Hotel&query_place_id=Ch...,22
190,BOD191,ChIJ7QtqF35xvhQRhO2gdmiBD64,Yalıkavak Marina Hotel by METT Collection,Yalıkavak,Bodrum,Muğla,Türkiye,Otel,NaN,4.80,5,267,832.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Merkez Mah, Yalıkavak, Çökertme Cd., 48990 Bodrum/Muğla, Türkiye",+902529700060,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Yal%C4%B1kavak%20Marina%20Hotel%20by%20METT%20Co...,22
191,BOD192,ChIJSQqN3GlxvhQRvBpAaSr4tEo,Yalıpark Beach Hotel,Yalıkavak,Bodrum,Muğla,Türkiye,Otel,NaN,4.30,5,647,128.00,Arama anındaki fiyat snapshot'ı; tarih/oda/kişi koşullarına göre değişebilir.,"Yalıkavak, Plaj Cd. NO:20/B, 48400 Bodrum/Muğla, Türkiye",+902523131111,NaN,2026-08-24,https://www.google.com/maps/search/?api=1&query=Yal%C4%B1park%20Beach%20Hotel&query_place_id=ChI...,22


### 5. Tek DataFrame içinde destinasyon kapsamı

Ayrı bölge özet dosyası yüklenmez. Her otelin bulunduğu bölgedeki tesis sayısı, ana `df` üzerinden `area_hotel_count` kolonuna hesaplanır. Böylece analiz boyunca tek çalışma tablosu kullanılır.

In [5]:
display(
    df[["area", "area_hotel_count"]]
    .drop_duplicates()
    .sort_values("area")
    .reset_index(drop=True)
)

,area,area_hotel_count
0,Akyarlar,16
1,Bitez,18
2,Bodrum Merkez,13
3,Göltürkbükü,10
4,Gümbet,15
5,Gümüşlük,15
6,Gündoğan,14
7,Güvercinlik,10
8,Kadıkalesi,4
9,Ortakent-Yahşi,21


### 6. Veri seti dokümantasyonu

README dosyası değiştirilmeden okunur. Yorum puanı, resmî yıldız sınıfı, fiyat snapshot'ı ve
veri toplama tarihi arasındaki kavramsal ayrımlar bu projenin sonraki aşamalarında korunacaktır.

In [6]:
README_PATH = file_paths["bodrum_hotels_master_README.txt"]
readme_text = README_PATH.read_text(encoding="utf-8")
display(Markdown(f"```text\n{readme_text}\n```"))

```text
BODRUM HOTELS MASTER DATASET
Toplanma tarihi: 2026-08-24

Benzersiz tesis sayısı: 192
Tekilleştirme anahtarı: Google Place ID

ÖNEMLİ:
- google_rating müşteri değerlendirme puanıdır (5 üzerinden); otelin yıldız sınıfı değildir.
- official_star_rating doğrulanmadığı kayıtlar için bilerek boş bırakılmıştır.
- search_price_usd_snapshot sabit otel fiyatı değildir; arama anındaki fiyat göstergesidir.
- Puan, yorum sayısı ve fiyat zamanla değişebileceğinden collected_at alanı saklanmıştır.
- Sonraki zenginleştirme: resmi yıldız/tesis sınıfı, oda-yatak sayısı, koordinatlar, amenities,
  Booking/Tripadvisor puanları ve bireysel müşteri yorumları.

```

README'nin kritik veri sözlüğü notları:

- `google_rating`, müşterilerin 5 üzerinden verdiği değerlendirme puanıdır; `official_star_rating` değildir.
- `official_star_rating`, yalnızca doğrulanmış resmî yıldız sınıfını ifade eder.
- `search_price_usd_snapshot`, sabit otel fiyatı değil, arama anındaki fiyat göstergesidir.
- `collected_at`, değişebilen puan, yorum ve fiyat bilgilerinin snapshot tarihini gösterir.

### 7. Veri seti şeması

Ana veri setindeki her kolon için veri tipi, dolu/eksik gözlem sayısı, benzersiz değer sayısı
ve ilk dolu örnek değer otomatik olarak özetlenir. Bu tablo Data Audit öncesi ilk genel bakıştır.

In [7]:
def first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if not values.empty else pd.NA

schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[column].dtype) for column in df.columns],
    "non_null_count": [int(df[column].notna().sum()) for column in df.columns],
    "null_count": [int(df[column].isna().sum()) for column in df.columns],
    "unique_count": [int(df[column].nunique(dropna=True)) for column in df.columns],
    "example_value": [first_non_null(df[column]) for column in df.columns],
})
schema

,column,dtype,non_null_count,null_count,unique_count,example_value
0,hotel_id,object,192,0,192,BOD001
1,place_id,object,192,0,192,ChIJAxKJBV4MvhQRxCb482IcG0M
2,hotel_name,object,192,0,192,Aksoy Taş Ev
3,area,object,192,0,14,Akyarlar
4,district,object,192,0,1,Bodrum
5,province,object,192,0,1,Muğla
6,country,object,192,0,1,Türkiye
7,property_category,object,192,0,13,Otel
8,official_star_rating,float64,0,192,0,<NA>
9,google_rating,float64,192,0,17,4.70


### 8. Data Collection özeti

- Ana CSV başarıyla yüklendi: **192 otel kaydı ve 19 kaynak kolonu**.
- `area_hotel_count` ana `df` içinden türetildi; çalışma tablosu **20 kolon** içeriyor.
- Ana veride **14 destinasyon/bölge** bulunuyor.
- Ayrı bölge özet dosyası analiz girdisi olarak kullanılmıyor.
- README dokümantasyonu mevcut; Excel kopyası yardımcı format olarak bulunuyor.
- Bu notebook ham veriyi değiştirmedi, eksikleri doldurmadı ve kayıt silmedi.
- Sonraki adım `02_data_audit.ipynb` ile eksiklik, benzersizlik, geçerlilik, tutarlılık ve kaynak kapsamını denetlemektir.